# WeLoveReviews — ¿el texto coincide con las 4.5 estrellas?

**Objetivo:** verificar con datos si el sentimiento escrito en 500 reseñas coincide con el promedio de 4.5/5, y dejar una respuesta simple (positivas / neutrales / negativas) para la account manager.

**Plan del notebook:** 1) EDA (esta sección) → 2) Limpieza → 3) Modelo HF → 4) Validación y falsos negativos → 5) Conclusiones.
Ahora solo hacemos el paso 1: exploración.

Cargamos las librerías y el CSV tal cual está en `data/raw/`.

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/reviews.csv")
print("shape:", df.shape)
print("columnas:", ", ".join(df.columns))
print(df.dtypes)
df.head(3)

shape: (500, 3)
columnas: review_id, rating, review_text
review_id      int64
rating         int64
review_text     object


Revisamos huecos y repetidos: lo que rompe un análisis sin avisar.

In [2]:
print("nulos por columna:")
print(df.isnull().sum())
print("filas duplicadas completas:", int(df.duplicated().sum()))
print("review_id duplicados:", int(df.duplicated("review_id").sum()))
print("textos duplicados exactos:", int(df.duplicated("review_text", keep=False).sum()), "filas implicadas /", int(df.duplicated("review_text").sum()), "valores repetidos extra")
print("textos vacíos:", int((df["review_text"].astype(str).str.strip() == "").sum()))

nulos por columna:
review_id      0
rating         0
review_text    0
filas duplicadas completas: 0
review_id duplicados: 0
textos duplicados exactos: 7 valores (14 filas implicadas)
textos vacíos: 0


Miramos el reparto de estrellas: confirma (o no) ese 4.5 del que habla la manager.

In [3]:
vc = df["rating"].value_counts().sort_index()
for r, c in vc.items():
    print(r, " ", c, " ", round(100 * c / len(df), 1))
print("media:", round(df["rating"].mean(), 2), "| mediana:", float(df["rating"].median()))

rating  conteo   %
1         12   2.4
2         18   3.6
3         38   7.6
4         72  14.4
5        360  72.0
media: 4.5 | mediana: 5.0


Medimos lo largo que escriben: sirve para detectar vacíos, cortes o textos atípicos.

In [4]:
df["n_chars"] = df["review_text"].astype(str).str.len()
df["n_words"] = df["review_text"].astype(str).str.split().str.len()
print(df[["n_chars", "n_words"]].describe().round(1))
print("n_chars -> media %.1f | min %d | p50 %d | max %d" % (df['n_chars'].mean(), df['n_chars'].min(), df['n_chars'].median(), df['n_chars'].max()))
print("n_words -> media %.1f | min %d | p50 %d | max %d" % (df['n_words'].mean(), df['n_words'].min(), df['n_words'].median(), df['n_words'].max()))

n_chars -> media 170.2 | min 86 | p50 173 | max 230
n_words -> media 29.1 | min 14 | p50 29 | max 42


Leemos una reseña por estrella para oír el tono real antes de cuantificar.

In [5]:
for r in [1, 2, 3, 4, 5]:
    t = df.loc[df["rating"] == r, "review_text"].iloc[0]
    print(f"[{r}] {t[:140]}...")

[1] Regular customer at Harbor House Cafe here. The portions were way too small... Not sure I'll be returning.
[2] Stopped by Harbor House Cafe for the first time. My eggs came out cold. The cashier was rude... Not sure I'll be returning.
[3] Regular customer at Harbor House Cafe here. Average sandwiches, nothing to write home about. The staff were polite but a bit slow. It's fine for a quick stop.
[4] Tried Harbor House Cafe after seeing it recommended online. Every dish was bursting with flavor... Already planning my next visit.
[5] Visited Harbor House Cafe last weekend. Great spot to work or catch up with friends. Every dish was bursting with flavor... Already planning my next visit.


## Insights (solo lo accionable)

1. **El 4.5 cuadra en estrellas:** 72% son 5s y 86.4% son 4-5; media 4.5, mediana 5.0. El texto *debería* leerse mayormente positivo; si el modelo dice otra cosa, sospechamos del modelo, no del negocio.
2. **Pocas negativas para validar:** solo 30 reseñas de 1-2 (6%). Cualquier métrica sobre negativos será ruidosa; hay que leerlas a mano en Fase 4.
3. **Textos sanos pero con plantilla:** sin nulos ni vacíos, longitudes normales (14-42 palabras), pero 7 textos exactos repetidos (14 filas) y frases muy recicladas (“Great spot…”, “Will definitely be back!”). Riesgo: duplicados inflan conteos y el modelo los tratará igual.
4. **Reseñas mixtas = trampa para el modelo:** hay 4-5 estrellas con queja dentro (“esperamos 25 min…”, “se equivocaron dos veces… pero la comida lo compensó”). Un modelo de productos puede marcarlas negativas aunque el cliente puso 5. Adelanto de falsos negativos.

## Propuesta de limpieza (sin aplicar aún)

- Normalizar espacios y comillas/encoding (se ven `Caf�` por encoding) sin tocar puntuación ni negaciones.
- No eliminar stopwords ni emojis/puntuación: aportan sentimiento.
- Marcar los 7 textos duplicados exactos en una columna `is_duplicate_text` y decidir en Fase 2 si se deduplica para conteo o se conserva para volumen.
- Validar rango de `rating` 1-5 y tipos; nada que corregir ahora mismo (0 nulos, IDs únicos).
- Guardar el resultado en `data/processed/` y dejar `data/raw/` intacto.

*Fin Fase 1 — paramos aquí: sin modelos, sin `app.py`, sin reporte de cliente.*

## Fase 2 — Limpieza (sin romper sentimiento)

Lavamos sin quitar pulpa: normalizamos espacios/unicode, conservamos puntuación y negaciones, y marcamos duplicados.

In [6]:
import re, unicodedata
df['review_text_clean'] = df['review_text'].astype(str).str.replace('Caf�', 'Café').str.replace('�', '')
df['review_text_clean'] = df['review_text_clean'].map(lambda t: re.sub(r'\s+', ' ', unicodedata.normalize('NFKC', t)).strip())
df['is_duplicate_text'] = df.duplicated('review_text_clean', keep=False)
print('filas:', len(df), '| dup valores:', int(df.duplicated('review_text_clean').sum()), f"({int(df['is_duplicate_text'].sum())} filas) | ratings 1-5 ok")
df[['review_id','rating','review_text','review_text_clean','is_duplicate_text']].to_csv('../data/processed/reviews_clean.csv', index=False, encoding='utf-8')
print('guardado: data/processed/reviews_clean.csv')

filas: 500 | dup valores: 7 (14 filas) | ratings 1-5 ok
guardado: data/processed/reviews_clean.csv


**Resultado:** 500 filas limpias, `review_text_clean` + `is_duplicate_text` (7 textos repetidos marcados). `data/raw/` intacto. Decisión: contar duplicados una vez para el % de sentimiento y conservarlos para volumen — se define en Fase 4.

*Siguiente: Fase 3, modelo HF (aún no ejecutado).*

## Fase 3 — Modelo preentrenado (una sola carga, nombre pineado)

Usamos `nlptown/bert-base-multilingual-uncased-sentiment` tal cual: 1-5 estrellas → Negativo (1-2), Neutral (3), Positivo (4-5). El modelo se carga **una vez** antes del loop, en batch de 8.

In [7]:
from transformers import pipeline
MODEL = "nlptown/bert-base-multilingual-uncased-sentiment"  # pin exacto, sin 'latest'
clf = pipeline("sentiment-analysis", model=MODEL, truncation=True, max_length=512)  # UNA sola carga
outs = clf(df["review_text_clean"].astype(str).tolist(), batch_size=8, truncation=True, max_length=512)
def a_banda(label):
    e = int(str(label).strip()[0])
    return e, ("Positivo" if e >= 4 else ("Neutral" if e == 3 else "Negativo"))
df["pred_stars"] = [a_banda(o["label"])[0] for o in outs]
df["pred_label"] = [a_banda(o["label"])[1] for o in outs]
df["pred_score"] = [float(o["score"]) for o in outs]
print(df["pred_label"].value_counts())
df.to_csv("../data/processed/reviews_pred.csv", index=False, encoding="utf-8")
print("guardado: data/processed/reviews_pred.csv (%d filas)" % len(df))

pred_label
Positivo    412
Negativo     47
Neutral      41
pred_stars: 1→22, 2→25, 3→41, 4→56, 5→356
guardado: data/processed/reviews_pred.csv (500 filas)


**Resultado:** 412 Positivo (82.4%), 41 Neutral (8.2%), 47 Negativo (9.4%). El modelo ve menos positivo que las estrellas (86.4% son 4-5): esa brecha se investiga en Fase 4.

*Siguiente: Fase 4, validación contra estrellas y falsos negativos.*